In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import __main__
import re
from pathlib import Path
from scipy.signal import butter, filtfilt, find_peaks, peak_widths, welch, peak_prominences 

# --- kvůli unpickle ---
class packet_merge:
    pass
__main__.packet_merge = packet_merge

FS = 2000.0
_TS_RE = re.compile(r"data_\d+_(\d{8})_(\d{6})\.npy$", re.IGNORECASE)

# ==========================================================
# DATABÁZE VLAKŮ (s rozvorem v mm pro výpočet rychlosti)
# ==========================================================
TRAIN_DB = [
    {"typ": "CZLoko1",            "pomer": 1.791667, "dvojkoli_mm": 2400},
    {"typ": "CZLoko2",            "pomer": 2.75,     "dvojkoli_mm": 2400},
    {"typ": "Škoda380",           "pomer": 2.48,     "dvojkoli_mm": 2500},
    {"typ": "ALSTOMTRAXX160",     "pomer": 2.988462, "dvojkoli_mm": 2600},
    {"typ": "ALSTOMTRAXX160B",    "pomer": 2.996154, "dvojkoli_mm": 2600},
    {"typ": "ALSTOMTRAXX140",     "pomer": 3.015385, "dvojkoli_mm": 2600},
    {"typ": "SIEMENSVectronDual", "pomer": 3.0,      "dvojkoli_mm": 2700},
    {"typ": "SIEMENSVectronCD",   "pomer": 2.166667, "dvojkoli_mm": 3000},
    {"typ": "SIEMENSVectron",     "pomer": 2.3,      "dvojkoli_mm": 3000},
    {"typ": "Škoda363",           "pomer": 1.59375,  "dvojkoli_mm": 3200},
    {"typ": "Pendolino",          "pomer": 6.037037, "dvojkoli_mm": 2700},
    {"typ": "LEO",                "pomer": 4.925926, "dvojkoli_mm": 2700},
    {"typ": "Panter",             "pomer": 6.916667, "dvojkoli_mm": 2400},
    {"typ": "Elefant",             "pomer": 6.3,     "dvojkoli_mm": 2600},    
    {"typ": "Newag Dragon 2",     "pomer": 1.00,     "dvojkoli_mm": 1950}
]

def parse_timestamp_from_filename(filename: str):
    m = _TS_RE.search(filename)
    if not m:
        return pd.NaT
    ymd, hms = m.group(1), m.group(2)
    return pd.to_datetime(ymd + hms, format="%Y%m%d%H%M%S", errors="coerce")

def process_one_file(filepath, threshold=100, min_distance_s=0.05,
                     prominence=None, width_rel_height=0.5, 
                     lowcut=1.0, highcut=50.0, filter_order=4,
                     cut_samples=300):
    filepath = Path(filepath)

    arr = np.load(filepath, allow_pickle=True)
    obj = arr.item()
    
    # Načtení celých dat
    seg = np.asarray(obj.chan_0_int_record, dtype=float)
    seg_vlt_raw = np.asarray(obj.chan_0_vlt_record, dtype=float)
    
    if len(seg) <= cut_samples:
        return pd.DataFrame()

    # =========================================================
    # OPRAVA: Nejdřív filtrace celých dat ('seg'), pak ořez
    # =========================================================
    b, a = butter(N=filter_order, Wn=[lowcut/(FS/2), highcut/(FS/2)], btype="bandpass")
    x_hp_celkem = filtfilt(b, a, seg) 

    # Oříznutí všech signálů o prvních 300 vzorků (zahodí se okrajový jev filtru)
    seg = seg[cut_samples:]
    seg_vlt_raw = seg_vlt_raw[cut_samples:]
    x_hp = x_hp_celkem[cut_samples:]
    # =========================================================

    # Zbytek kódu pokračuje stejně...
    distance = int(min_distance_s * FS)
    fp_kwargs = dict(height=threshold, distance=distance)
    if prominence is not None:
        fp_kwargs["prominence"] = prominence

    peaks, props = find_peaks(-x_hp, **fp_kwargs)

    if len(peaks) >= 4:
        time_s_all = peaks / FS
        loco_dt_12 = time_s_all[1] - time_s_all[0]
        loco_dt_23 = time_s_all[2] - time_s_all[1]
        loco_dt_34 = time_s_all[3] - time_s_all[2]
        loco_ratio = loco_dt_23 / loco_dt_12 if loco_dt_12 > 0 else np.nan
    else:
        loco_dt_12, loco_dt_23, loco_dt_34, loco_ratio = np.nan, np.nan, np.nan, np.nan

    if len(peaks) == 0:
        return pd.DataFrame()

    seg_vlt_no_dc = seg_vlt_raw - np.mean(seg_vlt_raw)
    frekvence_psd, psd_hodnoty = welch(seg_vlt_no_dc, fs=FS, nperseg=1024)
    maska_psd = (frekvence_psd >= 75) & (frekvence_psd <= 100)
    psd_75_100_mean = np.mean(psd_hodnoty[maska_psd])
    
    poskozeni = "Ano" if psd_75_100_mean > 1000 else "Ne"

    widths, _, _, _ = peak_widths(-x_hp, peaks, rel_height=width_rel_height)
    width_fwhm_s = peak_widths(-x_hp, peaks, rel_height=0.5)[0] / FS

    prom = peak_prominences(-seg, peaks)[0]

    real_peak_indices = peaks + cut_samples
    time_s = real_peak_indices / FS
    
    dt_prev_s = np.empty_like(time_s, dtype=float)
    dt_prev_s[0] = np.nan
    dt_prev_s[1:] = np.diff(time_s)

    ts = parse_timestamp_from_filename(filepath.name)

    df = pd.DataFrame({
        "file": filepath.name,
        "timestamp": ts,                 
        "peak_index": real_peak_indices.astype(int), 
        "time_s": time_s,                                            
        "dt_prev_s": dt_prev_s,
        "peak_depth": -x_hp[peaks],            
        "prominence": prom,           
        "width_s": widths / FS,
        "width_fwhm_s": width_fwhm_s,
        "psd_75_100_mean": psd_75_100_mean,
        "poskozeni_podvozku": poskozeni,
        "used_channel": "int_record",
        "loco_dt_12": loco_dt_12,
        "loco_dt_23": loco_dt_23,
        "loco_dt_34": loco_dt_34,
        "loco_ratio": loco_ratio 
    })

    return df

def classify_locomotive(row, base_tolerance=0.15, max_speed_kmh=170.0):
    # ==========================================================
    # NOVĚ: Ignorujeme klasifikaci, pokud jde o chybu měření
    # ==========================================================
    if row.get("chyba_mereni") == "Ano":
        return "chyba_měření"

    dt12 = row.get("loco_dt_12", np.nan)
    dt23 = row.get("loco_dt_23", np.nan) 
    dt34 = row.get("loco_dt_34", np.nan)
    measured_ratio = row.get("loco_ratio", np.nan) 

    if pd.isna(dt12) or pd.isna(measured_ratio) or dt12 <= 0:
        return "neurčen"
        
    is_coco = 0.85 <= measured_ratio <= 1.15

    if pd.notna(dt34):
        if is_coco:
            if pd.notna(dt23):
                symmetry_error = abs(dt12 - dt23) / dt12
                if symmetry_error > 0.15:
                    return "neurčen"
        else:
            symmetry_error = abs(dt12 - dt34) / dt12
            if symmetry_error > 0.15:
                return "neurčen"
            
    if is_coco and pd.notna(dt23):
        prumerny_cas_podvozku_s = (dt12 + dt23) / 2.0
    else:
        prumerny_cas_podvozku_s = (dt12 + dt34) / 2.0 if (pd.notna(dt34) and dt34 > 0) else dt12

    best_match = "neurčen"
    smallest_diff = float('inf')
    best_tolerance = base_tolerance 

    for train in TRAIN_DB:
        diff = abs(measured_ratio - train["pomer"])
        rozvor_m = train["dvojkoli_mm"] / 1000.0
        rychlost_kmh = (rozvor_m / prumerny_cas_podvozku_s) * 3.6
        
        current_tolerance = max(base_tolerance, train["pomer"] * 0.06)
        
        if rychlost_kmh <= max_speed_kmh:
            if diff < smallest_diff:
                smallest_diff = diff
                best_match = train["typ"]
                best_tolerance = current_tolerance 

    if smallest_diff > best_tolerance:
        return "neurčen"

    return best_match

def calculate_speed_kmh(row):
    # ==========================================================
    # NOVĚ: Ignorujeme výpočet rychlosti, pokud jde o chybu měření
    # ==========================================================
    if row.get("chyba_mereni") == "Ano":
        return np.nan

    typ = row.get("typ_vlaku", "neurčen")
    dt12 = row.get("loco_dt_12", np.nan)
    dt23 = row.get("loco_dt_23", np.nan)
    dt34 = row.get("loco_dt_34", np.nan)
    measured_ratio = row.get("loco_ratio", np.nan)
    
    if typ == "neurčen" or typ == "chyba_měření" or pd.isna(dt12) or dt12 <= 0:
        return np.nan
        
    is_coco = pd.notna(measured_ratio) and 0.85 <= measured_ratio <= 1.15
    
    if is_coco and pd.notna(dt23):
        prumerny_cas_podvozku_s = (dt12 + dt23) / 2.0
    else:
        prumerny_cas_podvozku_s = (dt12 + dt34) / 2.0 if (pd.notna(dt34) and dt34 > 0) else dt12

    for train in TRAIN_DB:
        if train["typ"] == typ:
            rozvor_m = train["dvojkoli_mm"] / 1000.0
            rychlost_ms = rozvor_m / prumerny_cas_podvozku_s
            return round(rychlost_ms * 3.6, 1)
            
    return np.nan

def process_folder(folder, pattern="*.npy", **kwargs):
    folder = Path(folder)
    files = sorted(folder.glob(pattern))

    all_df = []
    failed = []

    for f in files:
        try:
            df = process_one_file(f, **kwargs)
            if not df.empty:
                all_df.append(df)
        except Exception as e:
            failed.append((f.name, str(e)))

    db = pd.concat(all_df, ignore_index=True) if all_df else pd.DataFrame()

    if len(db) > 0:
        summary = (db.groupby("file")
                    .agg(
                        timestamp=("timestamp", "first"),
                        n_peaks=("peak_index", "count"),
                        first_t=("time_s", "min"),
                        last_t=("time_s", "max"),
                        mean_depth=("peak_depth", "mean"),
                        mean_dt_s=("dt_prev_s", "mean"),
                        psd_75_100_mean=("psd_75_100_mean", "first"),
                        poskozeni_podvozku=("poskozeni_podvozku", "first"), 
                        used_channel=("used_channel", "first"),
                        loco_dt_12=("loco_dt_12", "first"),
                        loco_dt_23=("loco_dt_23", "first"),
                        loco_dt_34=("loco_dt_34", "first"),
                        loco_ratio=("loco_ratio", "first")
                    ).reset_index())
        
        # ==========================================================
        # NOVĚ: Pořadí aplikováno - nejprve detekce chyby, pak zbytek
        # ==========================================================
        summary["chyba_mereni"] = summary["first_t"].apply(lambda x: "Ano" if x < 0.524 else "Ne")
        
        summary["typ_vlaku"] = summary.apply(classify_locomotive, axis=1)
        summary["rychlost_kmh"] = summary.apply(calculate_speed_kmh, axis=1)
        
    else:
        summary = pd.DataFrame()

    return db, summary, failed

# =========================
# POUŽITÍ: SPUŠTĚNÍ SKRIPTU
# =========================
folder = r"C:\Users\Petr Hadraba\OneDrive - VUT\Dokumenty\koleje\Koleje-Python\dataOlomouc" 

print("Začínám zpracování složky. Může to chvíli trvat...")
db, summary, failed = process_folder(
    folder,
    pattern="*.npy",
    threshold=170,        
    min_distance_s=0.05,  
    highcut=50.0          
)

# ==========================================
# FILTROVÁNÍ PODLE DATA 
# ==========================================
if not summary.empty:
    hraniční_datum = pd.to_datetime("2026-01-01")
    db = db[db["timestamp"] <= hraniční_datum]
    summary = summary[summary["timestamp"] <= hraniční_datum]

print("\n=================================")
print("ZPRACOVÁNÍ DOKONČENO")
print("=================================")
print(f"Úspěšně zpracováno souborů: {len(summary)}")

if not summary.empty:
    print("\nDETEKOVANÉ TYPY VLAKŮ:")
    print(summary["typ_vlaku"].value_counts().to_string())
    
    chyby = summary[summary["chyba_mereni"] == "Ano"]
    print(f"\nCHYBY MĚŘENÍ: Detekováno {len(chyby)} souborů s first_t pod 0.524s (vyřazeno z klasifikace).")
    
    poskozene_vlaky = summary[summary["poskozeni_podvozku"] == "Ano"]
    print(f"\nUPOZORNĚNÍ: Detekováno {len(poskozene_vlaky)} vlaků s možným poškozením podvozku (PSD > 1000)!")
    if not poskozene_vlaky.empty:
        print("Náhled detekovaných defektů:")
        for _, r in poskozene_vlaky.head(5).iterrows():
            cas = r['timestamp'].strftime('%H:%M:%S') if pd.notna(r['timestamp']) else "Neznámý"
            print(f"- Čas: {cas} | Soubor: {r['file']} | PSD: {r['psd_75_100_mean']:.0f} | Typ: {r['typ_vlaku']}")
    
    uspesne_rychlosti = summary[summary["rychlost_kmh"].notna()]
    print(f"\nRychlost se podařilo určit u {len(uspesne_rychlosti)} vlaků.")

# ==========================================================
# EXPORT
# ==========================================================
folder_p = Path(folder)
db.to_parquet(folder_p / "peaks_database_v2.parquet", index=False)
db.to_csv(folder_p / "peaks_database_v2.csv", index=False)
summary.to_parquet(folder_p / "peaks_summary_v2.parquet", index=False)
summary.to_csv(folder_p / "peaks_summary_v2.csv", index=False)
print(f"\nNové soubory (v2) uloženy do: {folder_p}")

Začínám zpracování složky. Může to chvíli trvat...


C:\Users\Petr Hadraba\AppData\Local\Temp\ipykernel_34580\3697284663.py:103: PeakPropertyWarning: some peaks have a prominence of 0
  prom = peak_prominences(-seg, peaks)[0]
C:\Users\Petr Hadraba\AppData\Local\Temp\ipykernel_34580\3697284663.py:103: PeakPropertyWarning: some peaks have a prominence of 0
  prom = peak_prominences(-seg, peaks)[0]



ZPRACOVÁNÍ DOKONČENO
Úspěšně zpracováno souborů: 1126

DETEKOVANÉ TYPY VLAKŮ:
typ_vlaku
SIEMENSVectronCD      322
Škoda363              237
Panter                190
LEO                   122
ALSTOMTRAXX160        110
Pendolino              69
ALSTOMTRAXX140         15
SIEMENSVectronDual     14
CZLoko2                11
chyba_měření            9
Škoda380                8
neurčen                 6
CZLoko1                 5
Elefant                 4
Newag Dragon 2          3
SIEMENSVectron          1

CHYBY MĚŘENÍ: Detekováno 9 souborů s first_t pod 0.524s (vyřazeno z klasifikace).

UPOZORNĚNÍ: Detekováno 114 vlaků s možným poškozením podvozku (PSD > 1000)!
Náhled detekovaných defektů:
- Čas: 14:27:38 | Soubor: data_02_20250825_142738.npy | PSD: 1419 | Typ: Škoda363
- Čas: 21:12:26 | Soubor: data_02_20250825_211226.npy | PSD: 1628 | Typ: LEO
- Čas: 20:54:38 | Soubor: data_02_20250826_205438.npy | PSD: 1137 | Typ: SIEMENSVectronCD
- Čas: 14:41:54 | Soubor: data_02_20250827_144154.npy | P